# **Limpieza de Datos**

En este archivo, se efectúa una limpieza precisa de los datos extraidos del sitio web de "National Health and Nutrition Examination Survey (NHNES)" y el resultado final que se va a usar para el entrenamiento de la red neuronal será guardado en un nuevo archivo creado, de formato de valores separados por comas (csv o "comma-separated values") 

In [1]:
import numpy as np
import pandas as pd

In [3]:
# 1. Cargar los archivos descargados (ajusta la ruta si están en una subcarpeta)
demo = pd.read_sas("../data/DEMO_J.XPT")
bmx = pd.read_sas("../data/BMX_J.XPT")
bpx = pd.read_sas("../data/BPX_J.XPT")
glu = pd.read_sas("../data/GLU_J.XPT")
ins = pd.read_sas("../data/INS_J.XPT")
hdl = pd.read_sas("../data/HDL_J.XPT")
trigly = pd.read_sas("../data/TRIGLY_J.XPT")

In [6]:
#2. Unir todas las tablas por el ID único del paciente, llamado SEQN
dfs = [demo, bmx, bpx, glu, ins, hdl, trigly]
df = dfs[0]
for next_df in dfs[1:]:
    df = pd.merge(df, next_df, on="SEQN", how="inner")

print(f"Registros tras el cruce inicial: {len(df)}")

Registros tras el cruce inicial: 3036


In [8]:
#3. Filtrado clínico: Adultos y con al menos 8 horas de ayuno
df = df[(df["RIDAGEYR"] >= 18) & (df["WTSAF2YR"] >= 8)].copy() 
#Es como si recortáramos el array pero aplicando esos filtros internos

In [9]:
# 4. Calcular presión arterial promedio (NHANES toma hasta 3 mediciones)
df["SBP"] = df[["BPXSY1", "BPXSY2", "BPXSY3"]].mean(axis=1)
df["DBP"] = df[["BPXDI1", "BPXDI2", "BPXDI3"]].mean(axis=1)

In [10]:
# 5. Seleccionar las variables clave y renombrarlas a nombres legibles
columnas = {
    "RIDAGEYR": "edad",
    "RIAGENDR": "sexo",  # 1 = Hombre, 2 = Mujer
    "BMXBMI": "imc",
    "BMXWAIST": "cintura_cm",
    "SBP": "presion_sistolica",
    "DBP": "presion_diastolica",
    "LBXGLU": "glucosa_ayunas",
    "LBDHDD": "colesterol_hdl",
    "LBXTR": "trigliceridos",
    "LBXIN": "insulina_ayunas",  # Solo temporal para el cálculo del target
}

In [11]:
df_clean = df[list(columnas.keys())].rename(columns=columnas)


In [12]:
# Eliminar filas que no tengan glucosa o insulina (sin ellas no hay etiqueta real)
df_clean = df_clean.dropna(subset=["glucosa_ayunas", "insulina_ayunas"]).copy()

In [13]:
# 6. Calcular HOMA-IR y crear la variable objetivo binaria (Corte clínico estándar: 2.5)
homa_ir = (df_clean["glucosa_ayunas"] * df_clean["insulina_ayunas"]) / 405
df_clean["insulino_resistente"] = (homa_ir > 2.5).astype(int)

In [14]:
# 7. ELIMINAR insulina_ayunas para evitar fuga de datos (Data Leakage)
df_final = df_clean.drop(columns=["insulina_ayunas"])

In [15]:
# Ajustar sexo a formato binario estándar (0 y 1)
df_final["sexo"] = df_final["sexo"].map({1: 1, 2: 0})

In [16]:
print(f"Dataset final listo: {df_final.shape} filas y {df_final.shape} columnas")

Dataset final listo: (2340, 10) filas y (2340, 10) columnas


In [17]:
print("\nDistribución de clases (0 = Sensible, 1 = Insulino-resistente):")


Distribución de clases (0 = Sensible, 1 = Insulino-resistente):


In [18]:
print(df_final["insulino_resistente"].value_counts(normalize=True).round(3))

insulino_resistente
1    0.524
0    0.476
Name: proportion, dtype: float64


In [19]:
# Guardar el CSV limpio para el siguiente notebook de entrenamiento
df_final.to_csv("nhanes_ir_clean.csv", index=False)